# Amazon Bedrock AgentCore Runtime에 A2A Agent 호스팅 - AWS IAM Inbound Authentication

## 개요

이 튜토리얼에서는 inbound authentication에 AWS IAM을 사용하여 A2A(Agent-to-Agent) agent를 Amazon Bedrock AgentCore Runtime에 호스팅하는 방법을 알아봅니다.

[A2A protocol](https://a2a-protocol.org/dev/specification/)은 독립적인 AI agent system 간 통신을 지원하도록 설계된 open standard입니다. A2A는 일반적으로 인증에 OAuth/JWT token을 사용하지만 AgentCore Runtime에서는 inbound request에 AWS IAM credentials를 구성할 수 있어 enterprise 보안 요구 사항을 충족합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Agent 호스팅                                              |
| Protocol            | A2A(Agent-to-Agent)                                       |
| 인증                | AWS IAM(SigV4)                                            |
| 튜토리얼 구성 요소  | IAM auth를 사용하여 A2A agent를 AgentCore Runtime에 호스팅|
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중급                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 Strands Agents     |

### 튜토리얼 아키텍처

이 튜토리얼에서는 IAM 인증을 사용하는 A2A agent를 AgentCore Runtime에 배포합니다.

시연을 위해 `greet_user`와 `get_agent_info`의 두 tool이 포함된 간단한 agent를 사용합니다.

### 튜토리얼 주요 기능

* Strands framework로 A2A agent 생성
* 로컬에서 A2A agent 테스트
* A2A agent를 Amazon Bedrock AgentCore Runtime에 호스팅
* IAM 인증(SigV4)을 사용하여 배포된 A2A agent 호출


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials 구성 완료
* Amazon Bedrock AgentCore SDK
* Strands Agents framework
* 실행 중인 Docker daemon

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import (
    destroy_bedrock_agentcore,
)
from boto3.session import Session
from pathlib import Path
import os

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)

agent_name = "a2a_agent_iam"

## A2A(Agent-to-Agent Protocol) 이해

A2A는 AI agent가 서로 통신할 수 있도록 지원하는 protocol입니다. 주요 개념은 다음과 같습니다.

* **Agent Card**: agent capability를 설명하는 metadata
* **Message Exchange**: agent 간 structured communication
* **Tools**: agent가 다른 agent에 노출할 수 있는 함수
* **IAM Authentication**: 안전한 인증을 위해 AWS SigV4 사용

AgentCore Runtime은 A2A agent가 기본 path인 `0.0.0.0:9000/`에 호스팅되기를 기대합니다.

### 프로젝트 구조

```
agentcore-a2a-iam-sample/
├── agent.py              # 주요 A2A agent 코드
├── client.py             # IAM auth를 사용하는 test client
├── requirements.txt      # 의존성
└── hosting_a2a_iam_auth.ipynb  # 이 Notebook
```

## Agent 코드 검토

배포할 agent 코드를 검토합니다.

In [ ]:
!cat agent.py

### 코드 동작 설명

* **Strands Agent**: Strands framework를 사용하여 agent 생성
* **@tool**: Python 함수를 agent tool로 변환하는 decorator
* **A2AServer**: A2A protocol을 지원하도록 agent 래핑
* **FastAPI**: agent용 HTTP server 제공
* **Tools**: agent capability를 보여주는 간단한 tool 두 개

## 선택 사항: 로컬 테스트

AgentCore Runtime에 배포하기 전에 agent를 로컬에서 테스트할 수 있습니다.

1. **Terminal 1**: agent 시작
   ```bash
   python agent.py
   ```
   
2. **Terminal 2**: agent card 테스트
   ```bash
   curl http://localhost:9000/.well-known/agent-card.json | jq .
   ```

3. **Terminal 2**: test message 전송
   ```bash
   curl -X POST http://localhost:9000 \
     -H "Content-Type: application/json" \
     -d '{
       "jsonrpc": "2.0",
       "id": "req-001",
       "method": "message/send",
       "params": {
         "message": {
           "role": "user",
           "parts": [{
             "kind": "text",
             "text": "Hello! What can you do?"
           }],
           "messageId": "test-001"
         }
       }
     }' | jq .
   ```

## AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 AgentCore Runtime 배포를 구성합니다. 다음 항목을 구성합니다.

* Entrypoint: `agent.py`
* execution role 자동 생성
* ECR repository 자동 생성
* Protocol: A2A
* Requirements 파일

configure 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

In [ ]:
print(f"Using AWS region: {region}")

required_files = ["agent.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="A2A",
    agent_name=agent_name,
)
print("Configuration completed ✓")

## AgentCore Runtime에 A2A Agent 시작

Dockerfile이 준비되었으므로 A2A agent를 AgentCore Runtime에 시작합니다. 이 과정에서 다음 작업을 수행합니다.

1. Amazon ECR repository 생성
2. Docker image build 및 push
3. AgentCore Runtime 생성
4. agent 배포

이 작업에는 몇 분 정도 걸릴 수 있습니다.

In [ ]:
print("Launching A2A agent to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## 배포된 A2A Agent 테스트

IAM 인증을 사용하는 client로 배포된 A2A agent를 테스트합니다.

client는 다음 작업을 수행합니다.
1. AWS credentials를 사용하여 SigV4로 request signing
2. agent card 가져오기
3. agent에 test message 전송
4. response 표시

In [ ]:
import os

os.environ["AGENT_ARN"] = launch_result.agent_arn

from client import *

# custom message로 테스트
custom_message = "Please greet me. My name is Bob."
await test_agent(launch_result.agent_arn, custom_message)

## IAM Authentication 이해

### 작동 방식

1. **Client Side**: client가 AWS credentials를 사용하여 SigV4로 HTTP request signing
2. **AgentCore Runtime**: IAM을 사용하여 signature 검증
3. **Agent**: 인증된 request 수신

### SigV4 인증 흐름

```python
# client가 SigV4 auth 생성
auth = SigV4HTTPXAuth(credentials, "bedrock-agentcore", region)

# Auth가 각 request를 자동으로 signing
async with httpx.AsyncClient(auth=auth) as client:
    response = await client.post(url, json=data)
```

전체 client 코드를 보려면 다음 셀을 실행합니다.

In [ ]:
!cat client.py

## 다음 단계

IAM 인증을 사용하는 A2A agent를 성공적으로 배포했으므로 다음 작업을 수행할 수 있습니다.

1. **Tool 추가**: agent에 추가 tool 확장
2. **Multi-Agent System**: 이 agent를 호출하는 orchestrator agent 생성
3. **Custom IAM Policy**: agent 액세스를 위한 세분화된 IAM policy 생성
4. **통합**: 다른 AWS service와 통합

### 관련 튜토리얼

* [IAM Auth로 MCP Server 호스팅](../02-hosting-MCP-server/hosting_mcp_server_iam_auth.ipynb)
* [JWT Auth로 A2A 호스팅](../05-hosting-a2a/01-a2a-getting-started-agentcore-strands.ipynb)
* [Multi-Agent System](../05-hosting-a2a/02-a2a-deploy-orchestrator.ipynb)

## 리소스 정리(선택 사항)

이 튜토리얼에서 생성한 리소스를 정리하려면 다음 셀을 실행합니다.

In [ ]:
destroy_bedrock_agentcore(config_path=Path(".bedrock_agentcore.yaml"))
print("✓ AgentCore Runtime resources deleted")

## 마무리

이 튜토리얼에서는 다음 방법을 학습했습니다.

* Strands framework를 사용하여 A2A agent 생성
* agent를 AgentCore Runtime에 배포
* inbound request에 IAM 인증 구성
* SigV4-signed request를 사용하여 agent 테스트